# QQQI / QQQ / TQQQ VIX v3 — aggressive recovery weight

Matched comparison of the frozen VIX v2 50% TQQQ state and a 75% TQQQ challenger. Signal and executed state traces must remain identical; only the leveraged-state weight may differ.

In [ ]:
from pathlib import Path

import yaml

from src.research.etf_rotation_experiment import fetch_adjusted_daily_bars
from src.research.strategy_experiment_journal import StrategyExperimentJournal
from src.research.vix_aggressive_tqqq_experiment import run_aggressive_tqqq_comparison
from src.research.vix_rotation_experiment import VIX_SYMBOL

In [ ]:
root = Path('..') if Path('../configs').exists() else Path('.')
baseline_path = root / 'configs/research_paradigms/qqqi_qqq_tqqq_vix_v2.yaml'
challenger_path = root / 'configs/research_paradigms/qqqi_qqq_tqqq_vix_v3_aggressive.yaml'
baseline_contract = yaml.safe_load(baseline_path.read_text(encoding='utf-8'))
challenger_contract = yaml.safe_load(challenger_path.read_text(encoding='utf-8'))
challenger_contract['change_control']

In [ ]:
symbols = [*challenger_contract['boundaries']['tradable_symbols'], VIX_SYMBOL]
bars, coverage = fetch_adjusted_daily_bars(
    symbols=symbols,
    start=challenger_contract['data']['start_date'],
    end=challenger_contract['data'].get('end_date'),
)
coverage

In [ ]:
metrics, results, prepared, diagnostics = run_aggressive_tqqq_comparison(
    bars, baseline_contract, challenger_contract
)
columns = [
    'total_return', 'cagr', 'annual_volatility', 'sharpe',
    'max_drawdown', 'calmar', 'turnover_units',
    'transaction_cost_paid', 'average_tqqq_weight',
]
metrics[[column for column in columns if column in metrics.columns]]

## Attribution check

The experiment is valid only when the close-decision and next-open position-state traces are identical.

In [ ]:
diagnostics

## Partial-leverage sessions

In [ ]:
base = results['rotation_vix_v2_50'].daily
aggressive = results['rotation_vix_v3_75'].daily
comparison = aggressive[[
    'position_state', 'vix_close', 'weight_QQQ', 'weight_TQQQ',
    'net_return', 'equity', 'drawdown'
]].copy()
comparison['baseline_net_return'] = base['net_return']
comparison.loc[comparison['position_state'].eq(2)].tail(30)

## Persistent run memory

The CLI runner writes a record under `artifacts/strategy_runs`. The notebook can inspect those records after a run.

In [ ]:
journal = StrategyExperimentJournal(root / 'artifacts/strategy_runs')
journal.latest(challenger_contract['experiment_id'])